
# Stock Sentiment Analysis with Agentics + DJIA News Dataset

This notebook demonstrates a **stock sentiment analysis** workflow using the classic Kaggle dataset:
- `Combined_News_DJIA.csv` (daily top 25 news headlines and a daily label indicating whether the **DJIA** went **Up (1)** or **Down (0)** the **next trading day**),
- `RedditNews.csv` (headline/time feed),
- `upload_DJIA_table.csv` (DJIA close data).

We show two paths:
1. **Baseline ML**: TF–IDF + Logistic Regression to predict daily DJIA up/down from headlines.
2. **Agentic LLM** (IBM **Agentics**, optional): Use an LLM to produce **structured sentiment** for each headline, aggregate per day, and evaluate against the daily DJIA label.

> ⚠️ The Agentics section requires configuring an LLM backend (e.g., **OpenAI** API key or a **local Ollama** endpoint). If you skip it, the baseline ML still works.


## 0. Setup

In [ ]:

# If you plan to use Agentics + OpenAI, uncomment the install and set OPENAI_API_KEY.
# If you plan to use a local LLM (Ollama), set OLLAMA_HOST and MODEL_NAME.

# !pip install -q agentics pydantic scikit-learn pandas numpy tqdm matplotlib requests

import os, sys, pandas as pd, numpy as np, re, json, math, datetime as dt
from pathlib import Path

DATA_DIR = Path("./dataset_extracted")  # adjust if needed
if not DATA_DIR.exists():
    # For this notebook's default layout, we expect files under ./dataset_extracted.
    # If you're running this elsewhere, set DATA_DIR to the folder containing the CSVs.
    DATA_DIR = Path("/mnt/data/dataset_extracted")
print("Using DATA_DIR =", DATA_DIR)

# Optional: configure OpenAI
# os.environ['OPENAI_API_KEY'] = 'sk-...'

# Optional: configure Ollama
# os.environ['OLLAMA_HOST'] = 'http://127.0.0.1:11434'
# MODEL_NAME = 'llama3'
MODEL_NAME = os.environ.get("AGENTICS_MODEL_NAME", "llama3")


## 1. Load & Inspect Data

In [ ]:

combined_path = DATA_DIR / "Combined_News_DJIA.csv"
reddit_path = DATA_DIR / "RedditNews.csv"
djia_path = DATA_DIR / "upload_DJIA_table.csv"

df = pd.read_csv(combined_path)
df['Date'] = pd.to_datetime(df['Date'])
print(df.shape)
df.head(3)


## 2. Preprocess Headlines

In [ ]:

# Combine Top1..Top25 into a long list per day
headline_cols = [c for c in df.columns if c.startswith("Top")]
daily = df[['Date','Label'] + headline_cols].copy()

# basic cleaning
def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s

for c in headline_cols:
    daily[c] = daily[c].astype(str).map(clean_text)

# Make a long dataframe: one row per (Date, headline)
long_rows = []
for _, row in daily.iterrows():
    date = row['Date']
    label = int(row['Label'])
    headlines = [row[c] for c in headline_cols]
    for h in headlines:
        if h and h != "nan":
            long_rows.append({"Date": date, "Label": label, "headline": h})
long_df = pd.DataFrame(long_rows)
print("Long headlines:", long_df.shape)
long_df.head(5)


## 3. Baseline ML: TF–IDF + Logistic Regression (Daily Label)

In [ ]:

from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import numpy as np

# Aggregate headlines per day into a single document
agg_df = long_df.groupby(['Date','Label'])['headline'].apply(lambda s: " \n ".join(s)).reset_index()
agg_df = agg_df.sort_values('Date').reset_index(drop=True)

# Train/test split by time
cut = int(len(agg_df)*0.8)
train_df = agg_df.iloc[:cut]
test_df = agg_df.iloc[cut:]

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=1000))
])

pipe.fit(train_df['headline'], train_df['Label'])
pred = pipe.predict(test_df['headline'])
proba = pipe.predict_proba(test_df['headline'])[:,1]

acc = accuracy_score(test_df['Label'], pred)
f1 = f1_score(test_df['Label'], pred)
try:
    auc = roc_auc_score(test_df['Label'], proba)
except:
    auc = float('nan')

print(f"Accuracy: {acc:.3f} | F1: {f1:.3f} | AUC: {auc:.3f}")
print("\nClassification report:\n", classification_report(test_df['Label'], pred, digits=3))


## 4. Agentic LLM Labeling with Agentics (Optional)


This section shows how to use **IBM Agentics** to produce **structured sentiment** for each headline, then
aggregate to a daily sentiment score and evaluate against the DJIA label.

You need:
- Either **OpenAI** credentials (`OPENAI_API_KEY`) or a **local Ollama** server (`OLLAMA_HOST`) with a model name.
- The `agentics` package installed.

We run on a **small sample** to control cost/time; adjust `N_DAYS` as desired.


In [ ]:

import os, math, pandas as pd
from typing import Optional

# Try importing Agentics
try:
    from pydantic import BaseModel, Field
    from agentics import Agentics as AG
    HAVE_AGENTICS = True
except Exception as e:
    print("Agentics not available:", e)
    HAVE_AGENTICS = False

# Define a typed output for agentic sentiment
if 'BaseModel' not in globals():
    from pydantic import BaseModel, Field

class SentimentDoc(BaseModel):
    label: str = Field(description="One of: Positive, Negative, Neutral")
    score: float = Field(description="A numeric polarity between -1 (very negative) and +1 (very positive)")
    rationale: str = Field(description="Short explanation focusing on financial/market impact")

def agentics_label_headlines(headlines):
    """Label a list of headlines using Agentics, return list[SentimentDoc].
    If Agentics is not available, fall back to a rule-based stub as placeholder.
    """
    results = []
    if HAVE_AGENTICS:
        try:
            # Minimal Agentics usage: create an agent with typed output and a financial sentiment instruction.
            agent = AG(
                atype=SentimentDoc,
                system="""You are a seasoned financial news sentiment analyst.
Return sentiment strictly for stock market impact. Output must obey the typed schema."""
            )
            # Use the << operator (logical transduction) if available, otherwise call agent directly
            try:
                agent = __import__('asyncio').get_event_loop().run_until_complete(agent << headlines)  # batch
                results = agent.states  # list[SentimentDoc]
            except Exception:
                # Fallback: map one by one synchronously
                for h in headlines:
                    a = __import__('asyncio').get_event_loop().run_until_complete(agent << [h])
                    results.append(a.states[0])
        except Exception as e:
            print("Agentics call failed, falling back to stub. Error:", e)
            HAVE_AGENTICS = False

    if not HAVE_AGENTICS:
        # Simple stub: keyword-based heuristic (for offline demo)
        POS = ['surge','beat','beats','record','profit','growth','optimistic','upgrade','rally','rise','soar','gain']
        NEG = ['fall','falls','plunge','loss','lawsuit','fraud','downgrade','cut','drop','decline','miss','slump','bankrupt']
        for h in headlines:
            hs = h.lower()
            sp = sum(w in hs for w in POS)
            sn = sum(w in hs for w in NEG)
            score = (sp - sn) / (sp + sn + 1e-6)
            if score > 0.15: label = "Positive"
            elif score < -0.15: label = "Negative"
            else: label = "Neutral"
            results.append(SentimentDoc(label=label, score=float(score), rationale="Heuristic keywords"))
    return results

# Choose a small sample of days to run agentically
N_DAYS = 50
sample = daily.sort_values('Date').head(N_DAYS)
sample_dates = sample['Date'].unique().tolist()

records = []
for d in sample_dates:
    rows = sample[sample['Date']==d]
    headlines = [rows[c].iloc[0] for c in [c for c in daily.columns if c.startswith('Top')]]
    sents = agentics_label_headlines(headlines)
    # aggregate: mean score, majority label
    mean_score = float(sum(s.score for s in sents) / max(len(sents),1))
    labels = [s.label for s in sents]
    maj = max(set(labels), key=labels.count)
    djia_label = int(rows['Label'].iloc[0])
    records.append({
        "Date": pd.to_datetime(d),
        "agentic_mean_score": mean_score,
        "agentic_majority": maj,
        "djia_label": djia_label
    })

agentic_df = pd.DataFrame(records).sort_values("Date")
agentic_df.head()


## 5. Evaluate Agentic Aggregate vs DJIA Daily Label

In [ ]:

from sklearn.metrics import roc_auc_score, accuracy_score

# Simple mapping: positive if mean_score > thresh
thresh = 0.0
pred_label = (agentic_df['agentic_mean_score'] > thresh).astype(int)
acc = accuracy_score(agentic_df['djia_label'], pred_label)
try:
    auc = roc_auc_score(agentic_df['djia_label'], agentic_df['agentic_mean_score'])
except Exception:
    auc = float('nan')

print(f"Agentic agg Accuracy: {acc:.3f} | AUC: {auc:.3f}")
agentic_df.head(10)


## 6. Plot Agentic Mean Score Over Time

In [ ]:

import matplotlib.pyplot as plt

plt.figure()
plt.plot(agentic_df['Date'], agentic_df['agentic_mean_score'], label='Agentic Mean Sentiment')
plt.axhline(0, linestyle='--')
plt.title('Agentic Mean Sentiment (First N Days)')
plt.xlabel('Date'); plt.ylabel('Sentiment Score')
plt.show()


## 7. Save Outputs

In [ ]:

out_path = Path("./agentic_results.csv")
agentic_df.to_csv(out_path, index=False)
print("Saved:", out_path.resolve())
